# pems03 按年节点增长数据处理

规则：
- pems03 从 **2001年3月4日** 开始。
- **2001年**：以第一天（2001-03-04）的 stations 作为该年的固定节点集合。
- **2002年及以后**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，得到随年份节点逐渐增长的数据集，每年保存为一个 CSV。

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [2]:
PEMS03_DIR = './pems03'
OUTPUT_DIR = '.'   # 每年 CSV 保存到当前目录，可改为例如 './pems03_yearly'
PREFIX = 'd03_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    "Timestamp", "Station", "District", "Freeway #",
    "Direction of Travel", "Lane Type", "Station Length",
    "Samples", "% Observed", "Total Flow", "Avg Occupancy", "Avg Speed"
]

## 1. 读取 pems03 目录，解析所有可用日期

In [3]:
def list_pems03_dates(data_dir):
    """扫描目录，返回 (日期 -> 文件名) 的字典。"""
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        # d03_text_station_5min_2001_03_04.txt.gz
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems03_dates(PEMS03_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f"pems03 共 {len(available)} 个日期文件")
print(f"首日: {first_date}, 末日: {sorted_dates[-1]}")

pems03 共 9066 个日期文件
首日: 2001-03-04, 末日: 2025-12-31


## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)


def read_station_ids_from_file(path):
    """从某日的 station 5min 文件中读取所有出现的 Station ID（去重）。"""
    df = pd.read_csv(
        path,
        header=None,
        usecols=[1],  # Station 列
        names=['Station'],
        compression='gzip'
    )
    return np.sort(df['Station'].dropna().unique().astype(int))

# 为每一年确定锚定日，并读取该日的节点集合
years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}   # year -> anchor date
year_nodes = {}    # year -> array of station IDs (sorted)

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f"警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年")
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f"{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}")

2001 年 锚定日=2001-03-04, 节点数=174
2002 年 锚定日=2002-01-01, 节点数=174
2003 年 锚定日=2003-01-01, 节点数=174
2004 年 锚定日=2004-01-01, 节点数=272
2005 年 锚定日=2005-01-01, 节点数=314
2006 年 锚定日=2006-01-01, 节点数=313
2007 年 锚定日=2007-01-01, 节点数=582
2008 年 锚定日=2008-01-01, 节点数=567
2009 年 锚定日=2009-01-01, 节点数=743
2010 年 锚定日=2010-01-01, 节点数=942
2011 年 锚定日=2011-01-01, 节点数=1018
2012 年 锚定日=2012-01-01, 节点数=1100
2013 年 锚定日=2013-01-01, 节点数=1179
2014 年 锚定日=2014-01-01, 节点数=1211
2015 年 锚定日=2015-01-01, 节点数=1254
2016 年 锚定日=2016-01-01, 节点数=1271
2017 年 锚定日=2017-01-01, 节点数=1277
2018 年 锚定日=2018-01-01, 节点数=1287
2019 年 锚定日=2019-01-01, 节点数=1247
2020 年 锚定日=2020-01-01, 节点数=1213
2021 年 锚定日=2021-01-01, 节点数=1297
2022 年 锚定日=2022-01-01, 节点数=1454
2023 年 锚定日=2023-01-01, 节点数=1640
2024 年 锚定日=2024-01-01, 节点数=1797
2025 年 锚定日=2025-01-01, 节点数=1859


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [5]:
def read_one_day(path, columns=None):
    """读取单日 12 列数据。"""
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(
        path,
        header=None,
        usecols=columns,
        names=[COLUMN_NAMES[i] for i in columns],
        compression='gzip'
    )
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime("%m/%d/%Y %H:%M:%S")
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    anchor = year_anchor[year]
    node_set = set(year_nodes[year])
    
    # 该年应包含的日期：首年从 first_date 到 12-31，其余从当年 01-01 到 12-31
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    
    if not chunks:
        print(f"{year} 年无可用数据，跳过")
        continue
    
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems03_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}")

2001: 节点数=174, 行数=15179760, 已保存 -> .\pems03_2001.csv
2002: 节点数=174, 行数=18286704, 已保存 -> .\pems03_2002.csv
2003: 节点数=174, 行数=18128928, 已保存 -> .\pems03_2003.csv
2004: 节点数=272, 行数=28498596, 已保存 -> .\pems03_2004.csv
2005: 节点数=314, 行数=32932188, 已保存 -> .\pems03_2005.csv
2006: 节点数=313, 行数=31612608, 已保存 -> .\pems03_2006.csv
2007: 节点数=582, 行数=55392024, 已保存 -> .\pems03_2007.csv
2008: 节点数=567, 行数=58944864, 已保存 -> .\pems03_2008.csv
2009: 节点数=743, 行数=77175624, 已保存 -> .\pems03_2009.csv
2010: 节点数=942, 行数=97780164, 已保存 -> .\pems03_2010.csv
2011: 节点数=1018, 行数=106580316, 已保存 -> .\pems03_2011.csv
2012: 节点数=1100, 行数=115287024, 已保存 -> .\pems03_2012.csv
2013: 节点数=1179, 行数=123365640, 已保存 -> .\pems03_2013.csv
2014: 节点数=1211, 行数=127132572, 已保存 -> .\pems03_2014.csv
2015: 节点数=1254, 行数=131774854, 已保存 -> .\pems03_2015.csv
2016: 节点数=1271, 行数=133664556, 已保存 -> .\pems03_2016.csv
2017: 节点数=1277, 行数=134221848, 已保存 -> .\pems03_2017.csv
2018: 节点数=1287, 行数=127793612, 已保存 -> .\pems03_2018.csv
2019: 节点数=1247, 行数=127033362, 

## 4. 简要查看各年节点数变化（随年份增长）

In [6]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
})
summary = summary.sort_values('year').reset_index(drop=True)
summary

,year,anchor_date,n_nodes
0,2001,2001-03-04,174
1,2002,2002-01-01,174
2,2003,2003-01-01,174
3,2004,2004-01-01,272
4,2005,2005-01-01,314
5,2006,2006-01-01,313
6,2007,2007-01-01,582
7,2008,2008-01-01,567
8,2009,2009-01-01,743
9,2010,2010-01-01,942


In [7]:
## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

# 读取每年的 `pems03_{year}.csv`，pivot 成：
# - **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
# - **列**：每个 Station ID
# - **值**：Total Flow

# 输出保存为 `pems03_{year}_flow.csv`

In [10]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(2001, 3, 4)

# 从目录中扫描所有 pems03_{year}.csv，提取年份列表
_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, 'pems03_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(r'pems03_(\d{4})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    """
    读取 pems03_{year}.csv，pivot 成 (时间步 x station) 的 Total Flow 矩阵，
    补全完整 5min 时间索引后保存。
    """
    src_path = os.path.join(output_dir, f'pems03_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    # 生成完整 5min 时间索引，补全缺失时间步
    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'pems03_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 25 个年度 CSV: 2001~2025
pivot_yearly_csv 函数已定义


In [11]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 2001 年 ...
  读取完成: 15,179,760 行
  2001: stations=174, 时间步=87264(期望87264), 全NaN行=24, 已保存 -> .\pems03_2001_flow.csv
  耗时: 14.1s

处理 2002 年 ...
  读取完成: 18,286,704 行
  2002: stations=174, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems03_2002_flow.csv
  耗时: 18.3s

处理 2003 年 ...
  读取完成: 18,128,928 行
  2003: stations=174, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems03_2003_flow.csv
  耗时: 17.2s

处理 2004 年 ...
  读取完成: 28,498,596 行
  2004: stations=272, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems03_2004_flow.csv
  耗时: 27.6s

处理 2005 年 ...
  读取完成: 32,932,188 行
  2005: stations=314, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems03_2005_flow.csv
  耗时: 32.8s

处理 2006 年 ...
  读取完成: 31,612,608 行
  2006: stations=313, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems03_2006_flow.csv
  耗时: 32.1s

处理 2007 年 ...
  读取完成: 55,392,024 行
  2007: stations=582, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems03_2007_flow.csv
  耗时: 55.7s

处理 2008 年 ...
  读取完成: 58,944,864 行
  2008: stations=567, 时间步=105408(期望105408)

In [12]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,2001,87264,174
1,2002,105120,174
2,2003,105120,174
3,2004,105408,272
4,2005,105120,314
5,2006,105120,313
6,2007,105120,582
7,2008,105408,567
8,2009,105120,743
9,2010,105120,942


In [13]:
# 抽样检查：读取 2001 年的 flow 文件看看前几行
check = pd.read_csv('./pems03_2001_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"pems03_2001_flow.csv 前 10 行, shape={check.shape}")
check

pems03_2001_flow.csv 前 10 行, shape=(10, 174)


,311831,311832,311844,311845,311847,311864,311903,311930,311972,311973,...,313172,313178,313184,313190,313197,313204,313329,313339,313344,313349
Timestamp,,,,,,,,,,,,,,,,,,,,,
2001-03-04 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:35:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2001-03-04 00:40:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
